# FLAG 2027 — NB-41: T3ENC-01, the Track 3 encoders on the VoxCeleb2 samples of FLAG_37

**Why.** Twin of FLAG_40 (same four encoders on our MAV-Celeb rows). With these, the local experiment can train a
face <-> voice head on MAV-Celeb + ~4,900 VoxCeleb2 identities in the encoder space of Track 3's VoxBlink system, and
measure whether the encoders and a head, without VoxBlink data, already reach its level (plan V1).

**Samples:** exactly those of FLAG_37, read from its key lists (`face_meta.csv`, `voice_meta.csv` in
`flag2027-feats-vox2`): face = middle frame of the same VoxCeleb2 face track, voice = the same clip, 12 s centre crop.
The output rows follow the key lists, so they align with FLAG_37's arrays.

| array | model | dim |
|---|---|---|
| `face_farl16` | FaRL ViT-B/16 LAION-Face 20M ep16 | 512 |
| `face_clipL` | `openai/clip-vit-large-patch14` @ 32bd642, image features | 768 |
| `voice_sam100p` | WeSpeaker `voxblink2_samresnet100` (VoxBlink2 pretrained) | 256 |
| `voice_sam100f` | WeSpeaker `voxblink2_samresnet100_ft` (VoxBlink2 -> **VoxCeleb2** fine-tune: these speakers are its training speakers) | 256 |

**Rules:** as FLAG_37: frozen encoders; no MAV-Celeb identity is compared with VoxCeleb2; Bengali: VoxCeleb2 may contain some
(organiser question 6); every clip keeps its key for a later language filter. Declare VoxCeleb2 in the system description.

**Inputs (Add Input):** **`flag2027-vox2-keys`** (the two key lists, 2.3 MB, upload of `kaggle_upload/flag2027-vox2-keys/`) OR the FLAG_37 output `flag2027-feats-vox2`: only `face_meta.csv` / `voice_meta.csv` are read. Optional: this
notebook's previous output, to resume. **Settings:** GPU T4, **Internet ON**.

**Time:** about 3-4 h: faces ~1 h (29k face tracks by HTTP range reads, two models); voices ~2-3 h (the same tars as FLAG_37,
streamed until every key is found, two models). Resumable per chunk / tar.

**Output:** `/kaggle/working/feats_vox2t3/` -> dataset **`flag2027-feats-vox2t3`** (about 0.2 GB, float16).

In [1]:
import os, sys, io, re, time, json, types, shutil, zipfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
CACHE = Path(os.environ.get("FLAG_CACHE", "/tmp/flag_cache"))           # checkpoints stay out of the output
ECAPA_DIR = Path(os.environ.get("FLAG_ECAPA", str(CACHE / "ecapa_sb")))
OUT = WORKDIR / "feats_vox2t3"
for d in (WORKDIR, CACHE, OUT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# ------------------------------------------------------------------ what to run (edit here)
ENCODERS = ["arcface", "ecapa", "vitage", "w2v2ag", "ibv", "iba"]   # ibv right before iba: one ImageBind load
EXT_SOURCES = ["v1_complete", "v2_complete"]                        # MAV-Celeb rows for STRESS-01; [] = skip
EXT_ENCODERS = ["arcface", "ecapa"]                                 # their ImageBind / age exist (FLAG_10)
SPLIT_FILTER = None            # e.g. ["no_gender/English"]; None = every split found (train + every test list)
ZIP_PREFIXES = ("train_set", "dev_set", "eval", "test")             # organiser zips unpacked when not extracted
ONLY_UNDER = None              # e.g. "eval": keep only lists whose path contains this (dev and eval both attached)
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| output", OUT, "| cache", CACHE)

KAGGLE | input /kaggle/input | output /kaggle/working/feats_vox2t3 | cache /tmp/flag_cache


In [2]:
# WeSpeaker (commit 9fecd6c, as Track 3) and openai CLIP (commit d50d76d, FaRL's code base): --no-deps, their pins would
# replace Kaggle's torch; the few runtime deps by hand.
if not LOCAL:
    !pip -q install ftfy regex kaldiio silero-vad av 2>&1 | tail -1
    !pip -q install --no-deps "git+https://github.com/wenet-e2e/wespeaker.git@9fecd6cb4f47475d01761d87c826298dff4ef18c" "git+https://github.com/openai/CLIP.git@d50d76daa670286dd6cacf3bcd80b5e4823fc8e1" 2>&1 | tail -2
else:                                           # local smoke test: wespeaker + its deps in an isolated folder
    for p_ in reversed([p for p in os.environ.get("FLAG_T3_PATH", "").split(";") if p]):
        sys.path.insert(0, p_)
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, soundfile as sf, transformers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 63.2 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.16.1 | device cuda Tesla T4


## 1. Audio / image helpers (FLAG_12 cell)

In [3]:
# ============================================================================ audio + batching
TARGET_SR = 16000
AUDIO_STATS = {"files": 0, "resampled": 0, "multichannel": 0, "short": 0}
MIN_SEC = 0.1                  # shorter clips are tiled to 0.1 s (w2v2 / AudioCLIP STFT fail on a few ms; v4 clips are >= 2 s)
BROKEN = {"audio": [], "image": []}   # unreadable sources: scored from a stand-in, listed in the manifest


def _src_name(src):
    """Full path of a file source (base names repeat across splits); the archive member name of a zip stream."""
    if isinstance(src, (bytes, bytearray)):
        return f"<{len(src)} bytes>"
    return str(src) if isinstance(src, (str, Path)) else str(getattr(src, "name", None) or src)


def load_audio(src):
    """Mono float32 at 16 kHz as the organiser loads audio (librosa.load(sr=16000, mono=True): channel mean + soxr_hq).
    All v4 wavs are 16 kHz mono; MAV-Celeb v1 has 44.1 kHz files. `src` = path or file-like.
    A file that cannot be read (or is empty / non-finite) -> 1 s of silence, listed in BROKEN["audio"]."""
    try:
        w, sr = sf.read(src if hasattr(src, "read") else str(src), dtype="float32", always_2d=True)
        if w.size == 0 or not np.isfinite(w).all():
            raise ValueError(f"{w.shape[0]} samples, finite {bool(np.isfinite(w).all())}")
    except Exception as e:
        BROKEN["audio"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable audio -> 1 s of silence:", BROKEN["audio"][-1])
        return np.zeros(TARGET_SR, np.float32)
    AUDIO_STATS["files"] += 1
    if w.shape[1] > 1:
        AUDIO_STATS["multichannel"] += 1
        w = w.mean(1, keepdims=True)
    w = w[:, 0]
    if sr != TARGET_SR:
        import librosa
        w = librosa.resample(w, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
        AUDIO_STATS["resampled"] += 1
    if len(w) < int(MIN_SEC * TARGET_SR):
        AUDIO_STATS["short"] += 1
        w = np.tile(w, int(np.ceil(MIN_SEC * TARGET_SR / len(w))))[:int(MIN_SEC * TARGET_SR)]
    return w


def centre_crop(w, L):
    if len(w) <= L:
        return w
    s = (len(w) - L) // 2
    return w[s:s + L]


def pack_batches(durs, max_padded_sec=240.0, max_bs=32):
    """Longest first; a batch costs len(batch) * longest item after padding."""
    order = np.argsort(-np.asarray(durs), kind="stable")
    batches, cur = [], []
    for i in order:
        longest = durs[cur[0]] if cur else durs[i]
        if cur and ((len(cur) + 1) * longest > max_padded_sec or len(cur) >= max_bs):
            batches.append(cur); cur = []
        cur.append(int(i))
    if cur:
        batches.append(cur)
    return batches


def run_batched(items, fn, durs, max_padded_sec=240.0, max_bs=32, cpu_fn=None, label=""):
    """fn(list of items) -> list of outputs, in item order. A CUDA OOM halves the batch; a single item goes to CPU."""
    out = [None] * len(items)
    t0, done_n = time.time(), 0

    def go(idx):
        try:
            res = fn([items[i] for i in idx])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(idx) == 1:
                res = (cpu_fn or fn)([items[idx[0]]])
            else:
                h = len(idx) // 2
                go(idx[:h]); go(idx[h:])
                return
        for i, r in zip(idx, res):
            out[i] = r

    batches = pack_batches(durs, max_padded_sec, max_bs)
    for bi, b in enumerate(batches):
        go(b)
        done_n += len(b)
        if bi % 50 == 0 or done_n == len(items):
            el = time.time() - t0
            log(f"  {label} {done_n}/{len(items)}  {el:.0f}s  eta {el / done_n * (len(items) - done_n):.0f}s")
    return out


def cos_rows(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    return np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)


def amp(flag):
    return torch.autocast("cuda", dtype=torch.float16, enabled=bool(flag) and DEV.type == "cuda")


def read_rgb(src):
    """PIL RGB image; an image that cannot be read -> a black 224 x 224 image, listed in BROKEN["image"]."""
    from PIL import Image
    try:
        return Image.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src).convert("RGB")
    except Exception as e:
        BROKEN["image"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable image -> black image:", BROKEN["image"][-1])
        return Image.new("RGB", (224, 224))

## 2. The four Track 3 encoders (same cell as FLAG_40)

In [4]:
# ============================================================================ T3ENC-01 encoders: the encoder family of Track 3's VoxBlink system, frozen
# Track 3 (a teammate's VoxBlink pipeline, CB Bangla ng ~16 / g ~25) used these four encoders; here they run on OUR rows
# (MAV-Celeb v4 train, v1/v2, dev lists; and VoxCeleb2 in FLAG_41), so we can test locally whether the encoders and a
# cross-modal head carry the gain WITHOUT VoxBlink data (docs/PLAN_VOXBLINK_2026-10-07.md, V1). Every sample on its own.
#   farl16V   FaRL ViT-B/16 LAION-Face 20M **ep16** (github.com/FacePerceiver/FaRL release, 652 MB) on openai CLIP ViT-B/16
#             (CLIP preprocessing: 224 bicubic, centre crop); 512-d image embedding (our existing `farl` is the ep64 copy)
#   clipLV    openai/clip-vit-large-patch14 @ 32bd642 (transformers): get_image_features, 768-d
#   sam100pA  WeSpeaker voxblink2_samresnet100 (VoxBlink2 pretrained), official ModelScope zip (wenet/wespeaker_pretrained_models)
#   sam100fA  WeSpeaker voxblink2_samresnet100_ft (VoxBlink2 pretrained -> VoxCeleb2 large-margin fine-tune)
#             both: wespeaker fbank + CMN, int16-scaled waveform (as our wssim34 / wsr221 cells), 256-d
import zipfile as _zf

FARL16_URL = "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep16.pth"
FARL16_SIZE = 652165542
CLIPL_ID, CLIPL_REV = "openai/clip-vit-large-patch14", "32bd64288804d66eefd0ccbe215aa642df71cc41"
WESPK_TREE = "https://modelscope.cn/api/v1/datasets/wenet/wespeaker_pretrained_models/oss/tree"
SAM_ZIPS = {"sam100pA": ("voxblink2_samresnet100.zip", 186890339), "sam100fA": ("voxblink2_samresnet100_ft.zip", 186890839)}
T3_STATS = {}


def fetch(url, dest, size=None):
    dest = Path(dest)
    if dest.exists() and (size is None or dest.stat().st_size == size):
        return dest
    torch.hub.download_url_to_file(url, str(dest) + ".part", progress=False)
    Path(str(dest) + ".part").replace(dest)
    assert size is None or dest.stat().st_size == size, f"{dest.name}: {dest.stat().st_size} bytes, expected {size}"
    return dest


class FaRL16Face:
    kind = "face"

    def __init__(self):
        import clip
        m, self.pre = clip.load("ViT-B/16", device="cpu", download_root=str(CACHE / "clip"))
        sd = torch.load(fetch(FARL16_URL, CACHE / "FaRL-ep16.pth", FARL16_SIZE), map_location="cpu", weights_only=False)["state_dict"]
        vis = [k for k in m.state_dict() if k.startswith("visual.")]
        hit = [k for k in vis if k in sd and sd[k].shape == m.state_dict()[k].shape]
        assert len(hit) == len(vis), f"FaRL ep16 covers {len(hit)}/{len(vis)} visual tensors"
        m.load_state_dict(sd, strict=False)
        self.m = m.visual.float().to(DEV).eval()

    def prep(self, im):
        return self.pre(im)

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        return {"": self.m(x.to(DEV)).float().cpu().numpy()}


class CLIPLFace:
    kind = "face"

    def __init__(self):
        from transformers import CLIPModel, CLIPImageProcessor
        self.proc = CLIPImageProcessor.from_pretrained(CLIPL_ID, revision=CLIPL_REV)
        self.m = CLIPModel.from_pretrained(CLIPL_ID, revision=CLIPL_REV, torch_dtype=torch.float32).to(DEV).eval()

    def prep(self, im):
        return self.proc(images=im.convert("RGB"), return_tensors="pt").pixel_values[0]

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        f = self.m.get_image_features(pixel_values=x.to(DEV))
        f = f.pooler_output if hasattr(f, "pooler_output") else f          # transformers 5 may return an output object
        return {"": f.float().cpu().numpy()}


def wespeaker_dir(zip_name, size):
    """The official zip (ModelScope, as wespeaker's own hub does), unpacked without the __MACOSX entries."""
    d = CACHE / "wespeaker" / zip_name[:-4]
    if (d / "avg_model.pt").exists() and (d / "config.yaml").exists():
        return d
    d.mkdir(parents=True, exist_ok=True)
    import requests
    tree = requests.get(WESPK_TREE, timeout=120).json()["Data"]
    url = [it["Url"] for it in tree if it.get("Key") == zip_name][0]
    z = fetch(url, d / zip_name, size)
    with _zf.ZipFile(z) as zf:
        for mem in zf.namelist():
            if not mem.startswith("__MACOSX") and not mem.endswith("/"):
                (d / Path(mem).name).write_bytes(zf.read(mem))
    assert (d / "avg_model.pt").exists() and (d / "config.yaml").exists(), f"{zip_name}: unexpected content {zf.namelist()[:6]}"
    return d


class SimAMVoice:
    kind = "voice"

    def __init__(self, name):
        for mod, attr in (("wespeaker.frontend.s3prl", "S3prlFrontend"), ("wespeaker.frontend.whisper_encoder", "whisper_encoder"),
                          ("wespeaker.frontend.w2vbert", "W2VBertFrontend"), ("wespeaker.diar.umap_clusterer", "cluster"),
                          ("wespeaker.diar.extract_emb", "subsegment")):
            if mod not in sys.modules:                     # SSL frontends / diarisation helpers: not used by fbank models
                stub = types.ModuleType(mod); setattr(stub, attr, None); sys.modules[mod] = stub
        import wespeaker
        self.m = wespeaker.load_model(str(wespeaker_dir(*SAM_ZIPS[name])))
        self.m.set_device("cuda:0" if DEV.type == "cuda" else "cpu")
        T3_STATS[name] = {"params_M": round(sum(p.numel() for p in self.m.model.parameters()) / 1e6, 1)}

    @torch.no_grad()
    def __call__(self, w, use_amp=True):
        pcm = torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32) * 32768.0)[None]
        feats = self.m.compute_features(pcm, sample_rate=16000, cmn=True)
        out = self.m.model(feats.to(self.m.device))
        out = out[-1] if isinstance(out, tuple) else out
        return {"": out[0].reshape(-1).float().cpu().numpy()}


NEW = {"farl16V": FaRL16Face, "clipLV": CLIPLFace, "sam100pA": lambda: SimAMVoice("sam100pA"),
       "sam100fA": lambda: SimAMVoice("sam100fA")}


def build(name):
    return NEW[name]()


def release(keep_ib=False):
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


ARRAYS_NEW = {"farl16V": ("face", {"": "farl16"}), "clipLV": ("face", {"": "clipL"}),
              "sam100pA": ("voice", {"": "sam100p"}), "sam100fA": ("voice", {"": "sam100f"})}
MAIN_NEW = {k: "" for k in NEW}
RESHAPE = {}

## 3. FLAG_37 key lists, zip listings

In [5]:
# ============================================================================ the FLAG_37 key lists, zip listings, helpers
import tarfile, requests, struct, zlib, threading
import concurrent.futures as cf
from huggingface_hub import HfFileSystem

MAX_TARS = 779
CLIP_SEC = 12.0                                          # centre crop, as FLAG_37
THREADS = 16
SEED = 0                                                 # FLAG_37's tar order: RandomState(0).permutation(779)
SMOKE_IDS = int(os.environ.get("VOX_SMOKE_IDS", "0")) if LOCAL else 0
ZIPS = [2, 3, 4, 5, 6]
ZIP_PATH = "datasets/ProgramComputer/voxceleb/vox2/vox2_mp4_{}.zip"
ZIP_URL = "https://huggingface.co/datasets/ProgramComputer/voxceleb/resolve/main/vox2/vox2_mp4_{}.zip"
VOICE_URL = "https://huggingface.co/datasets/gaunernst/voxceleb2-dev-wds/resolve/main/voxceleb2-dev-{:04d}.tar"
FS = HfFileSystem()
FACE_PARTS, VOICE_PARTS = OUT / "parts_face", OUT / "parts_voice"
for d in (FACE_PARTS, VOICE_PARTS):
    d.mkdir(parents=True, exist_ok=True)
for prev in (IN.rglob("feats_vox2t3") if IN.exists() and not LOCAL else []):     # resume: earlier output attached
    for sub in ("parts_face", "parts_voice"):
        for f in (prev / sub).glob("*.npz"):
            if not (OUT / sub / f.name).exists():
                shutil.copy(f, OUT / sub / f.name)
    for f in prev.glob("vox2_mp4_*_listing.csv"):
        if not (OUT / f.name).exists():
            shutil.copy(f, OUT / f.name)
def get_retry(url, tries=4, **kw):
    for k in range(tries):
        try:
            r = requests.get(url, timeout=300, **kw); r.raise_for_status(); return r
        except Exception as e:
            log(f"  retry {k + 1}/{tries} {url.rsplit('/', 1)[-1]}: {type(e).__name__}: {e}"); time.sleep(10 * (k + 1))
    raise RuntimeError(f"cannot fetch {url}")


def zip_listing(k):
    """Central directory of vox2_mp4_<k>.zip (= FLAG_37): one row per face track (dev/mp4/<spk>/<video>/<utt>.mp4)."""
    f = OUT / f"vox2_mp4_{k}_listing.csv"
    if f.exists():
        return pd.read_csv(f, dtype={"video": str, "utt": str})
    with FS.open(ZIP_PATH.format(k), "rb", block_size=2 ** 19) as fh:
        rows = [(i.filename, i.header_offset, i.compress_size, i.compress_type)
                for i in zipfile.ZipFile(fh).infolist() if i.filename.endswith(".mp4")]
    L_ = pd.DataFrame(rows, columns=["name", "off", "csize", "ctype"])
    sp = L_.name.str.split("/")
    L_["spk"], L_["video"], L_["utt"], L_["zip"] = sp.str[-3], sp.str[-2], sp.str[-1].str[:-4], k
    L_.to_csv(f, index=False)
    return L_


fk = [p for p in IN.rglob("face_meta.csv") if "feats_vox2t3" not in str(p)] if IN.exists() else []
vk = [p for p in IN.rglob("voice_meta.csv") if "feats_vox2t3" not in str(p)] if IN.exists() else []
assert fk and vk, "attach flag2027-vox2-keys (or the FLAG_37 output flag2027-feats-vox2): its face_meta.csv / voice_meta.csv fix the samples"
FK = pd.read_csv(fk[0], dtype={"spk": str, "video": str, "utt": str})
VK = pd.read_csv(vk[0], dtype={"spk": str, "video": str, "utt": str})
if SMOKE_IDS:                                            # local smoke test: speakers whose kept clips sit in the first tar
    keyset, keep = set(zip(VK.spk, VK.video, VK.utt)), []
    with get_retry(VOICE_URL.format(int(np.random.RandomState(SEED).permutation(779)[0])), stream=True) as r:
        r.raw.decode_content = True
        for m in tarfile.open(fileobj=r.raw, mode="r|"):
            k_ = tuple(m.name[:-4].split("/")[-3:]) if m.name.endswith(".m4a") else None
            if k_ in keyset and k_[0] not in keep:
                keep.append(k_[0])
            if len(keep) >= SMOKE_IDS:
                break
    FK, VK = FK[FK.spk.isin(keep)].reset_index(drop=True), VK[VK.spk.isin(keep)].reset_index(drop=True)
    MAX_TARS = 1
log(f"keys: {len(FK)} faces / {FK.spk.nunique()} ids, {len(VK)} voices / {VK.spk.nunique()} ids (from {fk[0].parent})")

10:48:06 keys: 29251 faces / 4877 ids, 38345 voices / 4877 ids (from /kaggle/input/datasets/hthlin/flag2027-feats-vox2/feats_vox2)


## 4. Faces (same face tracks as FLAG_37); resumable per chunk

In [6]:
# ============================================================================ faces: the same middle frame of the same face track as FLAG_37
_TL, URLS = threading.local(), {}


def resolve(k):
    with requests.get(ZIP_URL.format(k), headers={"Range": "bytes=0-0"}, allow_redirects=True, timeout=120, stream=True) as r:
        return r.url


def middle_frame(mp4):
    import av
    with av.open(io.BytesIO(mp4)) as c:
        st = c.streams.video[0]; n = st.frames or 50
        last = None
        for k, fr in enumerate(c.decode(st)):
            last = fr
            if k >= n // 2:
                break
        return last.to_image().convert("RGB") if last is not None else None


def fetch_frame(job):
    k, off, csize, ctype = job
    if not hasattr(_TL, "s"):
        _TL.s = requests.Session()
    for attempt in range(4):
        try:
            r = _TL.s.get(URLS[k], headers={"Range": f"bytes={off}-{off + 30 + 1024 + csize}"}, timeout=120)
            if r.status_code in (401, 403):
                URLS[k] = resolve(k); continue
            b = r.content
            n, e = struct.unpack("<HH", b[26:30])
            d = b[30 + n + e:30 + n + e + csize]
            return middle_frame(zlib.decompress(d, -15) if ctype == 8 else d)
        except Exception:
            time.sleep(2 * (attempt + 1))
    BROKEN["image"].append(f"vox2_mp4_{k} @ {off}")
    return None


LST = pd.concat([zip_listing(k) for k in ZIPS], ignore_index=True)
for k in ZIPS:
    URLS[k] = resolve(k)
SEL = FK[["spk", "video", "utt"]].merge(LST[["spk", "video", "utt", "zip", "off", "csize", "ctype"]], how="left",
                                         on=["spk", "video", "utt"])
assert SEL.zip.notna().all(), f"{SEL.zip.isna().sum()} face keys not found in the zip listings"
SEL["row"] = np.arange(len(SEL)); CH = 2400
log(f"faces: {len(SEL)} tracks, {len(SEL) // CH + 1} chunks, {THREADS} threads")
FACE_ENC = None
for c0 in range(0, len(SEL), CH):
    part = FACE_PARTS / f"faces_{c0 // CH:03d}.npz"
    if part.exists():
        continue
    if FACE_ENC is None:
        FACE_ENC = {"farl16": build("farl16V"), "clipL": build("clipLV")}
    t0 = time.time(); nb0 = len(BROKEN["image"])
    S_ = SEL.iloc[c0:c0 + CH]
    with cf.ThreadPoolExecutor(THREADS) as ex:
        imgs = list(ex.map(fetch_frame, zip(S_.zip.astype(int), S_.off.astype(int), S_.csize.astype(int), S_.ctype.astype(int))))
    ok = [i for i, im in enumerate(imgs) if im is not None]
    rows_ok, imgs = S_.row.values[ok], [imgs[i] for i in ok]
    R = {n: [] for n in FACE_ENC}
    for s in range(0, len(imgs), 64):
        b = imgs[s:s + 64]
        for n, enc in FACE_ENC.items():
            R[n].append(enc(torch.stack([enc.prep(im) for im in b]))[""])
    np.savez(part, row=rows_ok, **{n: np.concatenate(v).astype(np.float16) for n, v in R.items() if v})
    log(f"  chunk {c0 // CH}: {len(imgs)} faces, {len(BROKEN['image']) - nb0} failed [{time.time() - t0:.0f}s]")
FACE_ENC = None
release()

10:48:48 faces: 29251 tracks, 13 chunks, 16 threads


/usr/local/lib/python3.13/dist-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
100%|████████████████████████████████████████| 335M/335M [00:03<00:00, 103MiB/s]


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

10:54:20   chunk 0: 2400 faces, 0 failed [284s]
10:59:08   chunk 1: 2400 faces, 0 failed [288s]
11:03:59   chunk 2: 2400 faces, 0 failed [292s]
11:13:04   chunk 3: 2400 faces, 0 failed [545s]
11:22:23   chunk 4: 2400 faces, 0 failed [559s]
11:31:17   chunk 5: 2400 faces, 0 failed [534s]
11:40:14   chunk 6: 2400 faces, 0 failed [538s]
11:48:58   chunk 7: 2400 faces, 0 failed [524s]
11:57:46   chunk 8: 2400 faces, 0 failed [528s]
12:06:46   chunk 9: 2400 faces, 0 failed [541s]
12:15:28   chunk 10: 2400 faces, 0 failed [522s]
12:23:58   chunk 11: 2400 faces, 0 failed [509s]
12:25:41   chunk 12: 451 faces, 0 failed [104s]


## 5. Voices (same clips as FLAG_37); resumable per tar

In [7]:
# ============================================================================ voices: the same clips as FLAG_37 (12 s centre crop), streamed
def decode_m4a(b):
    import av
    out = []
    with av.open(io.BytesIO(b)) as c:
        res = av.AudioResampler(format="flt", layout="mono", rate=TARGET_SR)
        for fr in c.decode(c.streams.audio[0]):
            out += [o.to_ndarray().reshape(-1) for o in res.resample(fr)]
        out += [o.to_ndarray().reshape(-1) for o in res.resample(None)]
    return np.concatenate(out).astype(np.float32) if out else np.zeros(TARGET_SR // 10, np.float32)


row_of = {(s, v, u): i for i, (s, v, u) in enumerate(zip(VK.spk, VK.video, VK.utt))}
done = set()
for p in sorted(VOICE_PARTS.glob("tar*.npz")):            # resume
    done |= set(np.load(p)["row"].tolist())
order = list(np.random.RandomState(SEED).permutation(779))
VOICE_ENC = None
L = int(CLIP_SEC * TARGET_SR)
for t in order[:MAX_TARS]:
    if len(done) >= len(row_of):
        log("every voice key found"); break
    part = VOICE_PARTS / f"tar{t:04d}.npz"
    if part.exists():
        continue
    t0 = time.time(); waves, rows = [], []
    try:
        with get_retry(VOICE_URL.format(t), stream=True) as r:
            r.raw.decode_content = True
            tf = tarfile.open(fileobj=r.raw, mode="r|")
            for m in tf:
                if not m.name.endswith(".m4a"):
                    continue
                key = tuple(m.name[:-4].split("/")[-3:])
                i = row_of.get(key)
                if i is None or i in done:
                    continue
                try:
                    w = decode_m4a(tf.extractfile(m).read())
                except Exception as e:
                    BROKEN["audio"].append(f"{m.name} | {type(e).__name__}: {e}"); continue
                waves.append(centre_crop(w, L)); rows.append(i)
    except Exception as e:
        log(f"!! tar {t}: {type(e).__name__}: {e} -> skipped"); continue
    if not waves:
        np.savez(part, row=np.array([], int)); continue
    if VOICE_ENC is None:
        VOICE_ENC = {"sam100p": build("sam100pA"), "sam100f": build("sam100fA")}
    R = {n: np.stack([enc(w)[""] for w in waves]) for n, enc in VOICE_ENC.items()}
    np.savez(part, row=np.array(rows), **{n: v.astype(np.float16) for n, v in R.items()})
    done |= set(rows)
    log(f"tar {t:04d}: {len(waves)} clips [{time.time() - t0:.0f}s] | keys found {len(done)}/{len(row_of)}")
VOICE_ENC = None
release()

{'data_type': 'shard', 'dataloader_args': {'batch_size': 128, 'drop_last': True, 'num_workers': 16, 'pin_memory': False, 'prefetch_factor': 8}, 'dataset_args': {'aug_prob': 0.6, 'fbank_args': {'dither': 1.0, 'frame_length': 25, 'frame_shift': 10, 'num_mel_bins': 80}, 'filter': True, 'filter_args': {'max_num_frames': 800, 'min_num_frames': 100}, 'num_frms': 200, 'resample_rate': 16000, 'sample_num_per_epoch': 0, 'shuffle': True, 'shuffle_args': {'shuffle_size': 2500}, 'spec_aug': False, 'spec_aug_args': {'max_f': 8, 'max_t': 10, 'num_f_mask': 1, 'num_t_mask': 1, 'prob': 0.6}, 'speed_perturb': True}, 'enable_amp': False, 'exp_dir': 'exp/samresnet100/', 'gpus': [0, 1], 'log_batch_interval': 100, 'loss': 'CrossEntropyLoss', 'loss_args': {}, 'margin_scheduler': 'MarginScheduler', 'margin_update': {'epoch_iter': 4265, 'final_margin': 0.2, 'fix_start_epoch': 40, 'increase_start_epoch': 20, 'increase_type': 'exp', 'initial_margin': 0.0, 'update_margin': True}, 'model': 'SimAM_ResNet100_ASP', '

## 6. Assemble in FLAG_37 row order, probe, manifest

In [8]:
# ============================================================================ assemble in key-list order (= FLAG_37 rows), probe, manifest
def assemble(parts, n_rows, names):
    out = {n: None for n in names}
    for p in sorted(parts):
        Z = np.load(p)
        if not len(Z["row"]):
            continue
        for n in names:
            if out[n] is None:
                out[n] = np.full((n_rows, Z[n].shape[1]), np.nan, np.float16)
            out[n][Z["row"]] = Z[n]
    return out


Fo = assemble(FACE_PARTS.glob("*.npz"), len(FK), ["farl16", "clipL"])
Vo = assemble(VOICE_PARTS.glob("tar*.npz"), len(VK), ["sam100p", "sam100f"])
cov = {}
for n, A in {**{f"face_{k}": v for k, v in Fo.items()}, **{f"voice_{k}": v for k, v in Vo.items()}}.items():
    if A is not None:
        np.save(OUT / f"{n}.npy", A); cov[n] = round(float(np.isfinite(A[:, 0].astype(np.float32)).mean()), 4)
FK.to_csv(OUT / "face_meta.csv", index=False); VK.to_csv(OUT / "voice_meta.csv", index=False)
log(f"coverage (share of key-list rows filled): {cov}")

from sklearn.metrics import roc_curve
def eer(s, y):
    fpr, tpr, _ = roc_curve(y, s); fnr = 1 - tpr; i = np.nanargmin(np.abs(fnr - fpr)); return 100 * (fpr[i] + fnr[i]) / 2


l2n = lambda X: X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
PROBE, rng = {}, np.random.RandomState(0)
for side, A, M_ in (("face_farl16", Fo["farl16"], FK), ("face_clipL", Fo["clipL"], FK), ("voice_sam100p", Vo["sam100p"], VK),
                    ("voice_sam100f", Vo["sam100f"], VK)):
    if A is None:
        continue
    ok = np.isfinite(A[:, 0].astype(np.float32)); X = l2n(np.nan_to_num(A.astype(np.float32)))
    by = {s: np.where((M_.spk.values == s) & ok)[0] for s in M_.spk.unique()}
    sp = [s for s, ix in by.items() if len(ix) > 1]
    if len(sp) < 3:
        continue
    pos = [tuple(rng.choice(by[s], 2, replace=False)) for s in rng.choice(sp, 3000)]
    neg = [(rng.choice(by[a]), rng.choice(by[b])) for a, b in rng.choice(sp, (3000, 2)) if a != b]
    pr = np.array(pos + neg); y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    PROBE[side] = round(float(eer(np.sum(X[pr[:, 0]] * X[pr[:, 1]], 1), y)), 2)
print("within-modality identity probe (EER %; sam100f: VoxCeleb2 = its fine-tune speakers, expect very low):", PROBE)
man = {"created": time.strftime("%Y-%m-%d %H:%M"), "faces": len(FK), "voices": len(VK), "coverage": cov, "probe": PROBE,
       "encoders": {"farl16": FARL16_URL, "clipL": f"{CLIPL_ID}@{CLIPL_REV}", "sam100p": SAM_ZIPS["sam100pA"][0],
                    "sam100f": SAM_ZIPS["sam100fA"][0]},
       "samples": "the key lists of FLAG_37 (flag2027-feats-vox2); rows aligned with its arrays",
       "broken": BROKEN, "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
       "rules": "no MAV-Celeb identity was compared with these corpora; clips keep their keys for a later language filter"}
(OUT / "feats_manifest.json").write_text(json.dumps(man, indent=1, default=str))
print(json.dumps({k: v for k, v in man.items() if k != "broken"}, indent=1, default=str))
print("NEXT: Save Version -> Output -> New Dataset -> name it flag2027-feats-vox2t3 (private). If the session stopped "
      "early, attach the output and run again: finished chunks / tars are skipped.")

15:51:16 coverage (share of key-list rows filled): {'face_farl16': 1.0, 'face_clipL': 1.0, 'voice_sam100p': 1.0, 'voice_sam100f': 1.0}
within-modality identity probe (EER %; sam100f: VoxCeleb2 = its fine-tune speakers, expect very low): {'face_farl16': 11.5, 'face_clipL': 12.6, 'voice_sam100p': 3.6, 'voice_sam100f': 3.35}
{
 "created": "2026-10-07 15:51",
 "faces": 29251,
 "voices": 38345,
 "coverage": {
  "face_farl16": 1.0,
  "face_clipL": 1.0,
  "voice_sam100p": 1.0,
  "voice_sam100f": 1.0
 },
 "probe": {
  "face_farl16": 11.5,
  "face_clipL": 12.6,
  "voice_sam100p": 3.6,
  "voice_sam100f": 3.35
 },
 "encoders": {
  "farl16": "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep16.pth",
  "clipL": "openai/clip-vit-large-patch14@32bd64288804d66eefd0ccbe215aa642df71cc41",
  "sam100p": "voxblink2_samresnet100.zip",
  "sam100f": "voxblink2_samresnet100_ft.zip"
 },
 "samples": "the key lists of FLAG_37 (flag2027-feats-vox2); rows a